# DeepSculpt v2.0 - Google Colab Training

Complete workflow for training on Google Colab with GPU acceleration.

**Steps:**
1. Enable GPU (Runtime → Change runtime type → GPU)
2. Clone repository from GitHub
3. Train model
4. Generate samples

## 1. Setup - Clone Repository

In [ ]:
# Clone repository (update YOUR_USERNAME)
!git clone https://github.com/YOUR_USERNAME/deepSculpt.git
%cd deepSculpt/deepsculpt

## 2. Imports and Setup

In [ ]:
import sys
import os
from pathlib import Path
import time
import json

# Add to path
sys.path.insert(0, str(Path.cwd()))

# Core imports
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Import DeepSculpt
from core.data.generation.pytorch_sculptor import PyTorchSculptor
from core.data.generation.pytorch_collector import PyTorchCollector
from core.models.model_factory import PyTorchModelFactory
from core.training.pytorch_trainer import GANTrainer, TrainingConfig

print("✅ Modules loaded")

## 3. Configuration (Colab Optimized)

In [ ]:
CONFIG = {
    'void_dim': 32,           # Higher resolution
    'num_samples': 100,       # More samples
    'model_type': 'simple',
    'noise_dim': 100,
    'color_mode': 0,
    'epochs': 20,             # More epochs
    'batch_size': 16,         # Larger batches
    'learning_rate': 0.0002,
    'beta1': 0.5,
    'beta2': 0.999,
    'num_eval_samples': 5,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'sparse_mode': False,
    'mixed_precision': True,  # Enable for speed
    'output_dir': '/content/output',
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")

## 4. Dataset Class

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, file_paths, device='cpu'):
        self.file_paths = file_paths
        self.device = device
    
    def __len__(self):
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        structure = torch.load(self.file_paths[idx])
        if structure.dim() == 3:
            structure = structure.unsqueeze(0)
        structure = structure.float()
        if structure.max() > 1.0:
            structure = structure / 255.0
        return structure.to(self.device)

## 5. Generate Dataset

In [ ]:
output_dir = Path(CONFIG['output_dir'])
data_dir = output_dir / 'data'
data_dir.mkdir(parents=True, exist_ok=True)

sculptor_config = {
    'void_dim': CONFIG['void_dim'],
    'edges': (1, 0.3, 0.5),
    'planes': (1, 0.3, 0.5),
    'pipes': (1, 0.3, 0.5),
    'grid': (1, 4),
    'step': 1,
}

collector = PyTorchCollector(
    sculptor_config=sculptor_config,
    output_format='pytorch',
    base_dir=str(data_dir),
    device=CONFIG['device'],
    sparse_mode=CONFIG['sparse_mode'],
    verbose=True,
)

print(f"🎨 Generating {CONFIG['num_samples']} samples...")
start = time.time()

sample_paths = collector.create_collection(
    num_samples=CONFIG['num_samples'],
    batch_size=CONFIG['batch_size'],
    dynamic_batching=False,
)

print(f"✅ Generated in {time.time()-start:.1f}s")

## 6. Create DataLoader

In [ ]:
dataset = SimpleDataset(sample_paths, device=CONFIG['device'])
dataloader = DataLoader(
    dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
)

print(f"Dataset: {len(dataset)} samples")
print(f"Batches: {len(dataloader)}")

## 7. Create Models

In [ ]:
factory = PyTorchModelFactory(device=CONFIG['device'])

generator = factory.create_gan_generator(
    model_type=CONFIG['model_type'],
    void_dim=CONFIG['void_dim'],
    noise_dim=CONFIG['noise_dim'],
    color_mode=CONFIG['color_mode'],
    sparse=CONFIG['sparse_mode'],
)

discriminator = factory.create_gan_discriminator(
    model_type=CONFIG['model_type'],
    void_dim=CONFIG['void_dim'],
    color_mode=CONFIG['color_mode'],
    sparse=CONFIG['sparse_mode'],
)

gen_info = factory.get_model_info(generator)
disc_info = factory.get_model_info(discriminator)

print(f"Generator: {gen_info['total_parameters']:,} params")
print(f"Discriminator: {disc_info['total_parameters']:,} params")

## 8. Training Setup

In [ ]:
log_dir = output_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)

training_config = TrainingConfig(
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    learning_rate=CONFIG['learning_rate'],
    mixed_precision=CONFIG['mixed_precision'],
    gradient_clip=1.0,
    checkpoint_freq=5,
    log_dir=str(log_dir),
    use_tensorboard=False,
    use_wandb=False,
    use_mlflow=False,
)

gen_optimizer = torch.optim.Adam(
    generator.parameters(),
    lr=CONFIG['learning_rate'],
    betas=(CONFIG['beta1'], CONFIG['beta2']),
)

disc_optimizer = torch.optim.Adam(
    discriminator.parameters(),
    lr=CONFIG['learning_rate'],
    betas=(CONFIG['beta1'], CONFIG['beta2']),
)

trainer = GANTrainer(
    generator=generator,
    discriminator=discriminator,
    gen_optimizer=gen_optimizer,
    disc_optimizer=disc_optimizer,
    config=training_config,
    device=CONFIG['device'],
    noise_dim=CONFIG['noise_dim'],
)

print("✅ Trainer ready")

## 9. Training Loop

In [ ]:
checkpoint_dir = output_dir / 'checkpoints'
checkpoint_dir.mkdir(exist_ok=True)

print(f"🚀 Training {CONFIG['epochs']} epochs...\n")
start = time.time()

all_metrics = {
    'gen_loss': [],
    'disc_loss': [],
    'disc_real_acc': [],
    'disc_fake_acc': [],
}

for epoch in range(CONFIG['epochs']):
    print(f"Epoch {epoch+1}/{CONFIG['epochs']}")
    
    epoch_metrics = trainer.train_epoch(dataloader)
    
    for key in all_metrics.keys():
        if key in epoch_metrics:
            all_metrics[key].append(epoch_metrics[key])
    
    print(f"  Gen: {epoch_metrics.get('gen_loss', 0):.4f}")
    print(f"  Disc: {epoch_metrics.get('disc_loss', 0):.4f}")
    
    if (epoch + 1) % training_config.checkpoint_freq == 0:
        checkpoint_path = checkpoint_dir / f'checkpoint_epoch_{epoch+1}.pth'
        trainer.save_checkpoint(
            str(checkpoint_path),
            epoch=epoch+1,
            metrics=epoch_metrics,
            is_best=False
        )
        print(f"  💾 Saved")
    print()

print(f"✅ Done in {time.time()-start:.1f}s")

## 10. Plot Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if all_metrics['gen_loss']:
    axes[0].plot(all_metrics['gen_loss'], label='Generator', marker='o')
    axes[0].plot(all_metrics['disc_loss'], label='Discriminator', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Losses')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

if all_metrics['disc_real_acc']:
    axes[1].plot(all_metrics['disc_real_acc'], label='Real', marker='o')
    axes[1].plot(all_metrics['disc_fake_acc'], label='Fake', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Discriminator')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'metrics.png', dpi=150)
plt.show()

## 11. Save Models

In [ ]:
models_dir = output_dir / 'models'
models_dir.mkdir(exist_ok=True)

torch.save(generator.state_dict(), models_dir / 'generator.pt')
torch.save(discriminator.state_dict(), models_dir / 'discriminator.pt')

with open(models_dir / 'config.json', 'w') as f:
    json.dump({
        'void_dim': CONFIG['void_dim'],
        'noise_dim': CONFIG['noise_dim'],
        'color_mode': CONFIG['color_mode'],
        'epochs': CONFIG['epochs'],
    }, f, indent=2)

print("✅ Models saved")

## 12. Generate Samples

In [ ]:
samples_dir = output_dir / 'samples'
samples_dir.mkdir(exist_ok=True)

generator.eval()
generated = []

with torch.no_grad():
    for i in range(CONFIG['num_eval_samples']):
        noise = torch.randn(1, CONFIG['noise_dim'], device=CONFIG['device'])
        sample = generator(noise)
        generated.append(sample.cpu())
        torch.save(sample.cpu(), samples_dir / f'sample_{i:03d}.pt')

print(f"✅ Generated {len(generated)} samples")

## 13. Visualize

In [ ]:
fig, axes = plt.subplots(1, len(generated), figsize=(5*len(generated), 5))

if len(generated) == 1:
    axes = [axes]

for i, sample in enumerate(generated):
    s = sample.squeeze().numpy()
    if s.ndim == 4:
        s = s[0]
    mid = s.shape[0] // 2
    axes[i].imshow(s[mid], cmap='viridis')
    axes[i].set_title(f'Sample {i+1}')
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(output_dir / 'samples.png', dpi=150)
plt.show()

## 14. Download Results (Optional)

In [ ]:
# Zip results for download
!zip -r /content/deepsculpt_results.zip {CONFIG['output_dir']}

from google.colab import files
files.download('/content/deepsculpt_results.zip')